
Базовые модели рекомендации авиаперелётов
LogReg, RandomForest, CatBoost (pointwise), KNN

Методология оценки: sampled evaluation
  - Для каждого события (перелёта) генерируется 1 позитивный + N_NEG негативных кандидатов
  - Модель ранжирует кандидатов по скору
  - Метрики вычисляются по этому ранжированному списку

Параметры зафиксированы для воспроизводимости.


In [ ]:
#!/usr/bin/env python3

In [ ]:
N_NEG = 25                
RANK_K = [3, 5, 7, 10]    
N_TEST = 2                
MIN_INTERACTIONS = N_TEST + 1  
RANDOM_STATE = 42
USER_SAMPLE = 20000

# Пути к данным
PATH_23 = r"C:\Users\fedor\Documents\Работа_проекты\Аэрофлот\Recommend_mod_rep_МК 2\data\raw\flights_with_zones_23.parquet"
PATH_24 = r"C:\Users\fedor\Documents\Работа_проекты\Аэрофлот\Recommend_mod_rep_МК 2\data\raw\flights_with_zones_24.parquet"


In [ ]:
import pandas as pd
import numpy as np
import random
import time
import gc
import warnings
from math import log2
from collections import defaultdict

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from category_encoders import TargetEncoder

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Импорты загружены.")
print(f"Параметры: USER_SAMPLE={USER_SAMPLE}, N_NEG={N_NEG}, RANK_K={RANK_K}")


In [18]:
def load_and_combine_flights(path_23, path_24):
    flights_23 = pd.read_parquet(path_23)
    flights_24 = pd.read_parquet(path_24)
    df = pd.concat([flights_23, flights_24], ignore_index=True)
    
    # Удаление замкнутых маршрутов (обратные рейсы в домашний аэропорт)
    df = df[df['AIP_ARVL'] != df['MAIN_AIRPORT']]
    
    # Удаление пропусков
    df = df.dropna(subset=['FRQTFLR_CARD_ID', 'CITY_ARVL', 'SCHD_DEPTR_DT'])
    df['SCHD_DEPTR_DT'] = pd.to_datetime(df['SCHD_DEPTR_DT'], errors='coerce')
    df = df.dropna(subset=['SCHD_DEPTR_DT'])
    
    # Сортировка
    df = df.sort_values(['FRQTFLR_CARD_ID', 'SCHD_DEPTR_DT']).reset_index(drop=True)
    
    print(f"Всего записей после очистки: {len(df):,}, "
          f"уникальных пользователей: {df['FRQTFLR_CARD_ID'].nunique():,}")
    return df

df = load_and_combine_flights(PATH_23, PATH_24)


Всего записей после очистки: 4,719,643, уникальных пользователей: 329,638


In [ ]:
all_users = df['FRQTFLR_CARD_ID'].unique()

sampled_users = np.random.choice(all_users, size=USER_SAMPLE, replace=False)
df = df[df['FRQTFLR_CARD_ID'].isin(sampled_users)].reset_index(drop=True)
print(f"Подвыборка пользователей: {len(sampled_users):,}")



In [ ]:
df['NEXT_CITY_ARVL'] = df.groupby('FRQTFLR_CARD_ID')['CITY_ARVL'].shift(-1)
df_events = df.dropna(subset=['NEXT_CITY_ARVL']).reset_index(drop=True)
df_events['TOT_AMT'] = pd.to_numeric(df_events['TOT_AMT'], errors='coerce').fillna(0)
df_events['RDMP_MILES_AMT'] = pd.to_numeric(df_events['RDMP_MILES_AMT'], errors='coerce').fillna(0)


df_events['PREV_DATE'] = df_events.groupby('FRQTFLR_CARD_ID')['SCHD_DEPTR_DT'].shift(1)
df_events['days_since_prev'] = (df_events['SCHD_DEPTR_DT'] - df_events['PREV_DATE']).dt.days.fillna(999)

# Кумулятивный счётчик визитов пользователя в город
def add_user_city_count(df_in):
    cnt = defaultdict(int)
    res = []
    for _, row in df_in.iterrows():
        key = (row['FRQTFLR_CARD_ID'], row['CITY_ARVL'])
        res.append(cnt[key])
        cnt[key] += 1
    return res

df_events['user_city_count'] = add_user_city_count(df_events)
print(f"Событий с известной следующей целью: {len(df_events):,}")


Событий с известной следующей целью: 133,445


In [ ]:
def leave_n_out_split(full_df, n_test=N_TEST, min_interactions=MIN_INTERACTIONS):
    df_sorted = full_df.sort_values(
        ['FRQTFLR_CARD_ID', 'SCHD_DEPTR_DT']
    ).reset_index(drop=False).rename(columns={'index': 'global_idx'})
    
    train_idx, test_idx = [], []
    for uid, grp in df_sorted.groupby('FRQTFLR_CARD_ID'):
        if len(grp) >= min_interactions:
            test_rows = grp.tail(n_test)
            train_rows = grp.iloc[:-n_test]
            train_idx.extend(train_rows.index.tolist())
            test_idx.extend(test_rows.index.tolist())
    
    train_df = df_sorted.loc[train_idx].reset_index(drop=True)
    test_df = df_sorted.loc[test_idx].reset_index(drop=True)
    
    print(f"Train events: {len(train_df):,}, Test events: {len(test_df):,}")
    print(f"Train users: {train_df['FRQTFLR_CARD_ID'].nunique():,}, "
          f"Test users: {test_df['FRQTFLR_CARD_ID'].nunique():,}")
    return train_df, test_df

train_df, test_df = leave_n_out_split(df_events)

In [ ]:
train_cities = train_df['CITY_ARVL'].unique().tolist()
print(f"Уникальных городов в TRAIN: {len(train_cities)}")


Train events: 114,437, Test events: 17,952
Train users: 8,976, Test users: 8,976
Уникальных городов в TRAIN: 112


In [ ]:
def make_candidate_set(pos, available_cities, n_neg, random_seed_add=0):
    other_cities = [c for c in available_cities if c != pos]
    if len(other_cities) <= n_neg:
        return [pos] + other_cities
    rng = np.random.RandomState(RANDOM_STATE + random_seed_add + hash(pos) % 10000)
    negatives = rng.choice(other_cities, size=n_neg, replace=False).tolist()
    return [pos] + negatives

In [ ]:
def build_pairwise_dataset(events_df, available_cities, n_neg=N_NEG, start_event_id=0):
    rows = []
    event_id = start_event_id
    
    # Метаданные городов (зоны)
    city_meta = events_df.groupby('CITY_ARVL').agg({'ARVL_ZONE': 'first'}).to_dict()['ARVL_ZONE']
    
    for idx, row in events_df.iterrows():
        pos = row['NEXT_CITY_ARVL']
        if pos not in available_cities:
            continue
        
        candidates = make_candidate_set(pos, available_cities, n_neg, random_seed_add=idx)
        for cand in candidates:
            rows.append({
                'event_id': event_id,
                'FRQTFLR_CARD_ID': row['FRQTFLR_CARD_ID'],
                'event_city': row['CITY_ARVL'],
                'cand_city': cand,
                'label': 1 if cand == pos else 0,
                'MAIN_AIRPORT': row['MAIN_AIRPORT'],
                'SEASON': row['SEASON'],
                'FLT_TYPE': row['FLT_TYPE'],
                'SVC_CLS_DESC': row['SVC_CLS_DESC'],
                'DEPTR_ZONE': row['DEPTR_ZONE'],
                'ARVL_ZONE_cand': city_meta.get(cand, 'UNK'),
                'TOT_AMT': row['TOT_AMT'],
                'RDMP_MILES_AMT': row['RDMP_MILES_AMT'],
                'days_since_prev': row['days_since_prev'],
                'user_city_count': row['user_city_count'],
            })
        event_id += 1
    
    return pd.DataFrame(rows)

In [ ]:
pair_train = build_pairwise_dataset(train_df, train_cities, n_neg=N_NEG, start_event_id=0)
pair_test = build_pairwise_dataset(test_df, train_cities, n_neg=N_NEG, start_event_id=len(train_df))
print(f"pair_train: {pair_train.shape}, pair_test: {pair_test.shape}")
gc.collect()


In [ ]:
te = TargetEncoder(cols=['event_city', 'cand_city'])

te.fit(pair_train[['event_city', 'cand_city']], pair_train['label'])

ohe_features = ['MAIN_AIRPORT', 'SEASON', 'FLT_TYPE', 'SVC_CLS_DESC', 'DEPTR_ZONE', 'ARVL_ZONE_cand']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(pair_train[ohe_features])

num_features = ['TOT_AMT', 'RDMP_MILES_AMT', 'days_since_prev', 'user_city_count']

In [ ]:
def prepare_X_y(pair_df, te, ohe):
    df_local = pair_df.copy().reset_index(drop=True)
    enc = te.transform(df_local[['event_city', 'cand_city']])
    df_local = pd.concat([df_local, enc.add_suffix('_te')], axis=1)
    X_num = df_local[num_features].values
    X_ohe = ohe.transform(df_local[ohe_features])
    X_te = df_local[['event_city_te', 'cand_city_te']].values
    X = np.hstack([X_num, X_ohe, X_te])
    y = df_local['label'].values
    return X, y

In [ ]:
X_train, y_train = prepare_X_y(pair_train, te, ohe)
X_test, y_test = prepare_X_y(pair_test, te, ohe)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# Масштабирование для KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cat_features = ['MAIN_AIRPORT', 'SEASON', 'FLT_TYPE', 'SVC_CLS_DESC',
                'DEPTR_ZONE', 'ARVL_ZONE_cand', 'event_city', 'cand_city']
def prepare_catboost_data(pair_df):
    features = cat_features + num_features
    X = pair_df[features].copy()
    y = pair_df['label'].values
    return X, y
X_train_cb, y_train_cb = prepare_catboost_data(pair_train)
X_test_cb, y_test_cb = prepare_catboost_data(pair_test)
cat_indices = [X_train_cb.columns.get_loc(c) for c in cat_features]


X_train shape: (2975362, 87), X_test shape: (466752, 87)


In [ ]:
def compute_ranking_metrics(pair_df, scores, n_neg=N_NEG, K_list=RANK_K):
    df_local = pair_df.copy().reset_index(drop=True)
    df_local['score'] = scores
    
    results = {K: {'Precision': 0.0, 'Recall': 0.0, 'NDCG': 0.0,
                    'MAP': 0.0, 'HR': 0.0, 'count': 0}
               for K in K_list}
    
    # Для Coverage: множество уникальных рекомендованных городов
    coverage_sets = {K: set() for K in K_list}
    all_cities = set(df_local['cand_city'].unique())
    
    for eid, block in df_local.groupby('event_id'):
        expected_size = n_neg + 1
        if len(block) != expected_size:
            continue
        
        sorted_block = block.sort_values('score', ascending=False)
        labels = sorted_block['label'].values
        cities = sorted_block['cand_city'].values
        
        for K in K_list:
            top_k_labels = labels[:K]
            top_k_cities = cities[:K]
            
            hits = int(top_k_labels.sum())
            
            # Precision@K
            precision_k = hits / K
            
            # Recall@K (1 релевантный на event → = 1.0 если найден, иначе 0.0)
            recall_k = 1.0 if hits > 0 else 0.0
            
            # HitRate@K (то же что Recall при 1 релевантном)
            hr = 1.0 if hits > 0 else 0.0
            
            # NDCG@K
            dcg = sum(top_k_labels[j] / log2(j + 2) for j in range(K))
            idcg = 1.0 / log2(2)  # идеальный DCG при 1 релевантном на позиции 1
            ndcg_k = dcg / idcg if idcg > 0 else 0.0
            
            # AP@K (Average Precision)
            ap = 0.0
            num_rel = 0
            for j in range(K):
                if top_k_labels[j] == 1:
                    num_rel += 1
                    ap += num_rel / (j + 1)
            ap = ap / 1.0  # делим на число релевантных (=1)
            
            # Coverage
            for c in top_k_cities:
                coverage_sets[K].add(c)
            
            results[K]['Precision'] += precision_k
            results[K]['Recall'] += recall_k
            results[K]['NDCG'] += ndcg_k
            results[K]['MAP'] += ap
            results[K]['HR'] += hr
            results[K]['count'] += 1
    
    # Усреднение
    final = {}
    for K in K_list:
        n = results[K]['count']
        if n == 0:
            final[K] = {m: 0.0 for m in ['Precision', 'Recall', 'NDCG', 'MAP', 'HitRate', 'Coverage']}
            continue
        final[K] = {
            'Precision': results[K]['Precision'] / n,
            'Recall': results[K]['Recall'] / n,
            'NDCG': results[K]['NDCG'] / n,
            'MAP': results[K]['MAP'] / n,
            'HitRate': results[K]['HR'] / n,
            'Coverage': len(coverage_sets[K]) / len(all_cities) if len(all_cities) > 0 else 0.0,
        }
        final[K]['n_events'] = n
    
    return final


In [ ]:
def print_metrics(name, metrics, K_list=RANK_K):
    print(f"\n{'='*70}")
    print(f"  {name} — Ranking Metrics (sampled, {N_NEG+1} candidates per event)")
    print(f"{'='*70}")
    header = f"{'K':>3} | {'Prec@K':>8} | {'Recall@K':>8} | {'NDCG@K':>8} | {'MAP@K':>8} | {'HR@K':>8} | {'Cov@K':>8}"
    print(header)
    print("-" * len(header))
    for K in K_list:
        m = metrics[K]
        print(f"{K:3d} | {m['Precision']:8.4f} | {m['Recall']:8.4f} | {m['NDCG']:8.4f} | "
              f"{m['MAP']:8.4f} | {m['HitRate']:8.4f} | {m['Coverage']:8.4f}")
    if 'n_events' in metrics[K_list[0]]:
        print(f"\nОценено событий: {metrics[K_list[0]]['n_events']:,}")

In [ ]:
all_results = {}
print("Логистическая регрессия")

t0 = time.time()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
lr_train_time = time.time() - t0
print(f"Время обучения: {lr_train_time:.1f} сек")
y_prob_lr = lr.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, y_prob_lr)
print(f"AUC: {auc_lr:.4f}")

# Ранжирующие метрики
metrics_lr = compute_ranking_metrics(pair_test, y_prob_lr)
print_metrics("Логистическая регрессия", metrics_lr)
all_results['LogReg'] = {'metrics': metrics_lr, 'train_time': lr_train_time, 'AUC': auc_lr}

del lr
gc.collect()


In [ ]:
# Random Forest

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
rf.fit(X_train, y_train)
rf_train_time = time.time() - t0
print(f"Время обучения: {rf_train_time:.1f} сек")

y_prob_rf = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f"AUC: {auc_rf:.4f}")

metrics_rf = compute_ranking_metrics(pair_test, y_prob_rf)
print_metrics("Random Forest", metrics_rf)
all_results['RandomForest'] = {'metrics': metrics_rf, 'train_time': rf_train_time, 'AUC': auc_rf}

# Feature importance
feature_names = (num_features +
                 list(ohe.get_feature_names_out(ohe_features)) +
                 ['event_city_te', 'cand_city_te'])
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\nТоп-10 признаков (Random Forest):")
print(importance_df.head(10).to_string(index=False))

del rf
gc.collect()


In [ ]:
#CatBoost (pointwise classifier)

from catboost import CatBoostClassifier

t0 = time.time()
cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    border_count=128,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE,
    verbose=100,
    eval_fraction=0.2,
    early_stopping_rounds=50,
)
cb_model.fit(X_train_cb, y_train_cb, cat_features=cat_indices)
cb_train_time = time.time() - t0
print(f"Время обучения: {cb_train_time:.1f} сек")

y_prob_cb = cb_model.predict_proba(X_test_cb)[:, 1]
auc_cb = roc_auc_score(y_test_cb, y_prob_cb)
print(f"AUC: {auc_cb:.4f}")

metrics_cb = compute_ranking_metrics(pair_test, y_prob_cb)
print_metrics("CatBoost (pointwise)", metrics_cb)
all_results['CatBoost'] = {'metrics': metrics_cb, 'train_time': cb_train_time, 'AUC': auc_cb}

# Feature importance
fi_cb = cb_model.get_feature_importance()
fi_names_cb = X_train_cb.columns.tolist()
importance_df_cb = pd.DataFrame({
    'feature': fi_names_cb,
    'importance': fi_cb
}).sort_values('importance', ascending=False)
print("\nТоп-10 признаков (CatBoost):")
print(importance_df_cb.head(10).to_string(index=False))

del cb_model
gc.collect()


In [ ]:
# KNN
t0 = time.time()
knn = KNeighborsClassifier(
    n_neighbors=15,
    weights='distance',
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)
knn_train_time = time.time() - t0
print(f"Время обучения (fit): {knn_train_time:.1f} сек")

t0_pred = time.time()
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]
knn_pred_time = time.time() - t0_pred
print(f"Время инференса: {knn_pred_time:.1f} сек")

auc_knn = roc_auc_score(y_test, y_prob_knn)
print(f"AUC: {auc_knn:.4f}")

metrics_knn = compute_ranking_metrics(pair_test, y_prob_knn)
print_metrics("KNN", metrics_knn)
all_results['KNN'] = {'metrics': metrics_knn, 'train_time': knn_train_time, 'AUC': auc_knn}

del knn
gc.collect()


In [ ]:
K_main = 10
header = f"{'Модель':<20} | {'Prec@10':>8} | {'Recall@10':>9} | {'NDCG@10':>8} | {'MAP@10':>8} | {'HR@10':>8} | {'Cov@10':>8} | {'AUC':>6} | {'t(сек)':>8}"
print(header)
print("-" * len(header))
for name, res in all_results.items():
    m = res['metrics'][K_main]
    print(f"{name:<20} | {m['Precision']:8.4f} | {m['Recall']:9.4f} | {m['NDCG']:8.4f} | "
          f"{m['MAP']:8.4f} | {m['HitRate']:8.4f} | {m['Coverage']:8.4f} | {res['AUC']:6.4f} | {res['train_time']:8.1f}")


In [ ]:
for K in RANK_K:
    print(f"\n--- K = {K} ---")
    header = f"{'Модель':<20} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'MAP':>8} | {'HR':>8} | {'Cov':>8}"
    print(header)
    print("-" * len(header))
    for name, res in all_results.items():
        m = res['metrics'][K]
        print(f"{name:<20} | {m['Precision']:8.4f} | {m['Recall']:8.4f} | {m['NDCG']:8.4f} | "
              f"{m['MAP']:8.4f} | {m['HitRate']:8.4f} | {m['Coverage']:8.4f}")
